In [11]:
import os 
import re
import pandas as pd
import numpy as np

units_df = pd.read_csv('tables/CAIA_Measurement_Unit_Profile.csv')

selected_labs_df = pd.read_csv('tables/OMOP_data_availability_filtered.tsv', sep='\t')
selected_labs_df['SIMPLE_CONCEPT_ID'] = (selected_labs_df['Concept_ID']
                                         .apply(lambda x : re.sub(r'\s+', ' ', re.sub(r'\D', ' ', x)).strip().split(' ')))

In [15]:
unique_lab_units = []
labs_concept_id_dict = selected_labs_df[['Component', 'SIMPLE_CONCEPT_ID']]
for idx, row in labs_concept_id_dict.iterrows():
    units_data_sub = units_df.loc[units_df['MEASUREMENT_CONCEPT_ID'].astype(str)
                                  .isin(row['SIMPLE_CONCEPT_ID'])].copy()
    units_data_sub.loc[:,'LAB_CATCH_ALL'] = row.copy()['Component']
    unique_lab_units.append(units_data_sub[['SITE_ID', 'LAB_CATCH_ALL', 'MEASUREMENT_CONCEPT_ID', 'MEASUREMENT_CONCEPT_NAME', 'UNIT_CONCEPT_ID', 'UNIT_CONCEPT_NAME', 'UNIT_CONCEPT_CODE', 
                                            'RECORD_COUNT', 'MEASUREMENT_RECORD_TOTAL', 'PCT_RECORD_WITHIN_MEASUREMENT', 'PERSON_COUNT', 'MEASUREMENT_PERSON_TOTAL', 'PCT_PERSON_WITHIN_MEASUREMENT']].drop_duplicates())
unique_lab_units = pd.concat(unique_lab_units)

lab_groups = unique_lab_units['LAB_CATCH_ALL'].unique()
# unit_df = unique_lab_units.loc[(unique_lab_units['PCT_RECORD_WITHIN_MEASUREMENT'] > 0.1) & 
#                                (unique_lab_units['PCT_PERSON_WITHIN_MEASUREMENT'] > 0.1)].sort_values(by=['MEASUREMENT_CONCEPT_NAME', 'SITE_ID'])
unit_df = unique_lab_units.sort_values(by=['MEASUREMENT_CONCEPT_NAME', 'SITE_ID'])

# unit_df.to_csv('tables/selected_lab_units.csv', index=False)
unique_lab_units.sort_values(by=['MEASUREMENT_CONCEPT_NAME', 'SITE_ID']).to_csv('tables/full_lab_units.csv', index=False)
# unit_df = pd.read_csv('tables/selected_lab_units.csv')

In [16]:
unique_units = unit_df[['LAB_CATCH_ALL', 'MEASUREMENT_CONCEPT_ID', 'UNIT_CONCEPT_ID', 'UNIT_CONCEPT_NAME', 'UNIT_CONCEPT_CODE']].drop_duplicates()

In [17]:
unique_units['UNIT_RECORD_COUNT'] = np.nan

for i in range(len(unique_units)):
    row = unique_units.iloc[i]
    record_count = unit_df.loc[(unit_df['LAB_CATCH_ALL'] == row['LAB_CATCH_ALL']) &
                               (unit_df['UNIT_CONCEPT_ID'] == row['UNIT_CONCEPT_ID']) &
                               (unit_df['UNIT_CONCEPT_NAME'] == row['UNIT_CONCEPT_NAME']), 'RECORD_COUNT'].sum()
    unique_units.iloc[i, unique_units.columns.get_loc('UNIT_RECORD_COUNT')] = record_count

In [18]:
unique_units.to_csv('tables/unit_counts_full.csv', index=False)

In [38]:
unique_units[['LAB_CATCH_ALL', 'MEASUREMENT_CONCEPT_ID']].drop_duplicates()

,LAB_CATCH_ALL,MEASUREMENT_CONCEPT_ID
170,Alanine Aminotransferase (ALT),3006923
741,Albumin,3024561
606,Albumin/Globulin ratio,3020509
915,Alkaline Phosphatase (ALP),3035995
243,Alpha-fetoprotein (AFP),3009306
404,Aspartate Aminotransferase (AST),3013721
822,Direct Bilirubin (conjugated),3027597
718,Total Bilirubin,3024128
927,Height,3036277
630,Temperature,3020891


In [13]:
unit_df.loc[(unit_df['LAB_CATCH_ALL'] == row['LAB_CATCH_ALL']) &
            (unit_df['UNIT_CONCEPT_ID'] == row['UNIT_CONCEPT_ID']) &
            (unit_df['UNIT_CONCEPT_NAME'] == row['UNIT_CONCEPT_NAME'])]

,SITE_ID,LAB_CATCH_ALL,MEASUREMENT_CONCEPT_ID,MEASUREMENT_CONCEPT_NAME,UNIT_CONCEPT_ID,UNIT_CONCEPT_NAME,UNIT_CONCEPT_CODE,RECORD_COUNT,MEASUREMENT_RECORD_TOTAL,PCT_RECORD_WITHIN_MEASUREMENT,PERSON_COUNT,MEASUREMENT_PERSON_TOTAL,PCT_PERSON_WITHIN_MEASUREMENT
170,DFCI,Alanine Aminotransferase (ALT),3006923,Alanine aminotransferase [Enzymatic activity/v...,0,(NULL/0),NaN,4431,3581870,0.1237,1653,220890,0.7483


In [8]:
curated_units = pd.read_csv('tables/measurements_and_units_to_include.csv')

In [9]:
curated_units['LAB_CATCH

,LAB_CATCH_ALL,UNIT_CONCEPT_ID,UNIT_CONCEPT_NAME,UNIT_CONCEPT_CODE,UNIT_RECORD_COUNT,STANDARD_UNIT,NEEDS_CONVERSION,ADDITION_FACTOR,MULTIPLICATIVE_FACTOR
0,Alanine Aminotransferase (ALT),0,(NULL/0),NaN,4431,8645,No,NaN,NaN
1,Alanine Aminotransferase (ALT),8645,unit per liter,[U]/L,9519140,8645,No,NaN,NaN
2,Alanine Aminotransferase (ALT),8923,international unit per liter,[iU]/L,73220,8645,No,NaN,NaN
3,Albumin,0,(NULL/0),NaN,3188631,8713,No,NaN,NaN
4,Albumin,8713,gram per deciliter,g/dL,9939497,8713,No,NaN,NaN
...,...,...,...,...,...,...,...,...,...
116,Testosterone,0,(NULL/0),NaN,45,8817,No,NaN,NaN
117,Blood Urea Nitrogen (BUN),0,(NULL/0),NaN,6803,8840,No,NaN,NaN
118,Blood Urea Nitrogen (BUN),8840,milligram per deciliter,mg/dL,12791426,8840,No,NaN,NaN
119,Activated partial thromboplastin time (aPTT),8555,second,s,2344706,8555,No,NaN,NaN
